<a href="https://colab.research.google.com/github/Ferphs961118/Workshop_UACh-PSU_Genomics/blob/Files/Genomica_Fitobacterias_UACh_PSU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Introducción al análisis genómico de bacterias fitopatógenas

En este cuaderno trabajaremos con datos reales de secuenciación de bacterias obtenidos mediante tecnología **Nanopore**. Cada estudiante recibirá un archivo de lecturas crudas en formato FASTQ para realizar las siguientes tareas:

- Ensamblado del genoma con **Flye**
- Reordenamiento del genoma para iniciar en el gen **dnaA**
- Anotación genómica con **Bakta**
- Evaluación de completitud con **BUSCO**
- Comparación genómica (ANI) entre cepas
- Análisis pan-genómico grupal con **Panaroo** (si aplica)

Este flujo representa un pipeline básico pero poderoso para estudiar la diversidad, el contenido génico y la calidad de genomas bacterianos.  
Todo el análisis se realizará desde **Google Colab**, sin necesidad de instalaciones locales.

> ⚠️ **Cada estudiante debe trabajar únicamente con el genoma que se le asignó.**

Ubicar tu archivo FASTQ dentro de Drive

Cada estudiante recibió su archivo .fastq.gz desde la carpeta compartida por Microsoft Teams.

Descarga el archivo .fastq.gz a tu computadora.
Súbelo a Google Drive, dentro de una carpeta personalizada (por ejemplo https://drive.google.com/file/d/**1VaiDdnwjnu2abv3bXd1pJo2MoKcRK-1R**/view?usp=share_link).
Anota bien el nombre de la carpeta, por ejemplo:

**Genomes**


## 📁 Paso 1. Montar tu Google Drive

Todos tus archivos deben estar almacenados en Google Drive, en una carpeta personal con un nombre intuitivo (ej. `Genoma-Workshop/`).  
Así, todos los resultados y archivos intermedios quedarán guardados automáticamente.

In [ ]:
from google.colab import drive
import os  # ✅ necesario para manejar rutas y carpetas
# Montar Google Drive (si no lo has hecho antes)
from google.colab import drive
drive.mount('/content/drive')

# Verifica el contenido de tu carpeta principal
!ls -lh /content/drive/MyDrive/

### 🔍 1. Verifica la ubicación de tu genoma en Drive

Una vez que hayas montado tu Google Drive, necesitas asegurarte de que el archivo `.fastq.gz` está en la carpeta correcta. En este ejemplo, usamos una carpeta llamada `Genomes`. Ejecuta el siguiente código para listar su contenido:

‼️Asegúrate de anotar el nombre exacto del archivo que vas a usar.

In [ ]:
# Ver contenido dentro de la carpeta Genomes
!ls -lh /content/drive/MyDrive/Genomes

### 📂 2. Define la ruta del archivo como variable de trabajo

Después de verificar el archivo, define su ruta completa en una variable. Este paso es necesario para poder usar ese archivo en los análisis posteriores:

In [ ]:
import os

# Ruta completa al archivo FASTQ en tu Drive
input_fastq = "/content/drive/MyDrive/Genomes/VRP0126B.fastq.gz"

# Verifica que el archivo exista
if os.path.exists(input_fastq):
    print(f"✅ Archivo encontrado: {input_fastq}")
else:
    print("❌ Archivo no encontrado. Verifica la ruta.")


In [4]:
# Verifica el contenido de tu unidad para ubicar la carpeta
!ls -lh /content/drive/MyDrive/

total 250M
drwx------ 2 root root 4.0K Jun 15  2022 'Colab Notebooks'
-rw------- 1 root root 270K Mar  4  2019 'Copia de Lista de plagas 2019 Vigilancia activa moscas exóticas cafe y cultivos estrategicos 16 01 2019 (1).xlsx'
-rw------- 1 root root 221K Feb 13  2020 'Copia de Lista de plagas 2019 Vigilancia activa moscas exóticas cafe y cultivos estrategicos 16 01 2019.xlsx'
-rw------- 1 root root  11M Sep  4  2018  fitohormonas.pptx
-rw------- 1 root root  51K Feb 14  2020 'FO-DFI-12 Inventario de materiales, reactivos e insumos sustanciales para el diagnóstico.xlsx'
-rw------- 1 root root  172 Oct 25  2024 'Formulario sin título.gform'
-rw------- 1 root root  29M Sep  4  2018 'Fungicidas Clasificación y Mecanismos de acción CP2007.ppt'
drwx------ 7 root root 4.0K May 28 01:40  Genomes
-rw------- 1 root root 3.8M Jan 11  2022  IMG_2218.JPG
-rw------- 1 root root 3.3M Jan 11  2022  IMG_3115.JPG
-rw------- 1 root root 3.4M Jan 11  2022  IMG_3117.JPG
-rw------- 1 root root 3.0M Jan

## ⚙️ ¿Qué es Conda y por qué lo necesitamos?

**Conda** es un sistema de gestión de entornos y paquetes. Nos permite instalar herramientas bioinformáticas como Flye, Bakta o BUSCO sin conflicto entre ellas.

Google Colab **no incluye Conda por defecto**, así que primero debemos instalarlo para poder trabajar con herramientas de bioinformática que están disponibles en **Bioconda** (una colección de paquetes específicos para ciencias de la vida).


In [ ]:
# 🔧 INSTALAR CONDA EN GOOGLE COLAB
# Esto solo se hace UNA VEZ al inicio del cuaderno
!pip install -q condacolab
import condacolab
condacolab.install()


## 🧱 2. Ensamblado con Flye

**Flye** es un ensamblador para lecturas largas como las de Oxford Nanopore. Reconstruye el genoma completo de un organismo a partir de lecturas sin ensamblar (FASTQ).

### 🛠 ¿Qué vamos a hacer?
- Instalar Flye (solo una vez)
- Ejecutar el ensamblado con tu archivo `.fastq.gz`


In [ ]:
# 🔧 INSTALAR FLYE
# (esto solo se hace una vez por entorno)
!conda install -y -c bioconda flye


In [ ]:
# 💻 EJECUTAR ENSAMBLADO
#Asegurate de cambiar /content/drive/MyDrive/Genomes/VRP0126B.fastq.gz y  /content/drive/MyDrive/Genomes/ensamblado por las rutas de entrada y salida de tus archivos
# Esto puede tardar entre  2-3 horas dependiendo del tamaño
!flye --nano-raw /content/drive/MyDrive/Genomes/VRP0126B.fastq.gz --out-dir /content/drive/MyDrive/Genomes/ensamblado --threads 2


## 📊 Evaluar la calidad del ensamblado: métricas clave

Después de ensamblar y anotar un genoma, es fundamental reportar **estadísticas básicas** que describan la calidad y completitud del ensamblado. Estas métricas no solo ayudan a interpretar tus resultados, sino que son **requeridas frecuentemente en publicaciones científicas y repositorios como NCBI**.

Aquí un ejemplo típico:

Total length: 3,129,330 bp

Fragments: 2

Fragments N50: 3,067,374 bp

Largest frg: 3,067,374 bp

Scaffolds: 0

Mean coverage: 384x


### 🔍 ¿Qué significa cada métrica y por qué es importante?

- **Total length:** tamaño total del genoma ensamblado. Ayuda a confirmar si corresponde con el tamaño esperado de la especie (~3 Mbp para muchas bacterias).
- **Fragments:** número de contigs (fragmentos ensamblados). Lo ideal es 1 (genoma completo). Un número bajo indica un ensamblado de buena calidad.
- **N50:** longitud mínima del contig más grande que cubre al menos el 50% del ensamblado. A mayor N50, más continuo es el ensamblado.
- **Largest fragment:** tamaño del contig más largo. Idealmente debería ser cercano al total si tienes un ensamblado circular o casi completo.
- **Scaffolds:** número de fragmentos unidos artificialmente con gaps. Para ensamblados sin scaffolding (como Flye), esto debe ser 0.
- **Mean coverage:** profundidad promedio de secuenciación. Un valor alto (como 384x) implica alta confiabilidad en la secuencia, aunque valores mayores a 100x ya son más que suficientes para la mayoría de análisis bacterianos.

Estas métricas **deben incluirse en tablas o en el texto de resultados** cuando reportes nuevos ensamblados o genomas comparativos. También sirven como **criterios de exclusión o inclusión** para análisis posteriores (comparaciones, ANI, pangenomas, etc.).


## ✅ 1. Evaluación de calidad con BUSCO

**BUSCO** (Benchmarking Universal Single-Copy Orthologs) busca genes esenciales que deberían estar presentes en cualquier genoma completo de un determinado linaje (en este caso, bacterias).

Esta herramienta responde a una pregunta clave:  
> ¿Qué tan completo es mi ensamblado genómico?

Se utiliza comúnmente en artículos científicos como medida estándar de calidad de ensamblados. El resultado incluye:

- **Complete BUSCOs (C):** genes completos encontrados
- **Duplicated (D):** presentes más de una vez
- **Fragmented (F):** genes parciales
- **Missing (M):** genes esenciales no encontrados

Un ensamblado de buena calidad debe tener >95% de BUSCOs completos y <5% faltantes.


In [ ]:
# 🔧 INSTALAR BUSCO
!conda install -y -c bioconda -c conda-forge busco

In [ ]:
# 💻 EJECUTAR BUSCO
#Asegurante de reemplazar "/content/drive/MyDrive/Genomes/ensamblado/assembly.fasta" con el directorio de salida que te dio Flye

!busco -i /content/drive/MyDrive/Genomes/ensamblado/assembly.fasta \
       -o busco_out \
       -l bacteria_odb10 \
       -m genome \
       -f


### 🧠 ¿Cómo interpretar esto?

- **C: 93.5%** de los genes esenciales esperados están presentes y completos → muy buen ensamblado.
- **S: 93.5%** son genes únicos (lo ideal en bacterias haploides).
- **D: 0%** no hay duplicaciones espurias → no hay sobreensamblado ni problemas con repeticiones.
- **F: 2.4%** genes fragmentados → posiblemente por bordes de contigs.
- **M: 4.0%** genes ausentes → aceptable si están cerca del 5%.

🟢 **Conclusión:** Este genoma tiene una calidad **alta** y es apto para anotación y análisis comparativo.

> Estas cifras deben reportarse en manuscritos científicos y presentaciones como evidencia de que el ensamblado es confiable.


## 🔄 Rotar el genoma ensamblado para iniciar en `dnaA`

Rotar el genoma para que comience en el gen **dnaA** es una práctica común en genómica bacteriana. Esto **facilita la anotación**, la comparación filogenética y el análisis pan-genómico, ya que estandariza el punto de inicio del genoma.

> ⚠️ **IMPORTANTE:**  
> La secuencia usada como referencia para `dnaA` en el script es específica de **ciertas bacterias Gram positivas**.  
> **Debes reemplazarla por la secuencia consenso de `dnaA` correspondiente a tu organismo** (puedes obtenerla desde GenBank, Bakta o Prokka si ya anotaste tu genoma).

Aunque esta rotación **no altera la información genética**, mejora su visualización y alineación en análisis comparativos.

**‼️SOLO DEBES ROTAR EL GENOMA, SI OBTUVISTE 1 CONTIG (1 CROMOSOMA), O SI TIENES 1 CROMOSOMA Y PLÁSMIDO(S)**

In [ ]:

#Asegurante de reemplazar "/content/drive/MyDrive/Genomes/ensamblado/assembly.fasta" con el directorio de salida que te dio Flye
# Asegurate de reemplazar "/content/drive/MyDrive/Genomes/ensamblado/genoma_rotado.fasta" con el directorio y nombre de tu archivo para tu genoma rotado

from Bio import SeqIO, Seq
from Bio.SeqRecord import SeqRecord

# Secuencia consenso de dnaA (puede variar entre especies)
start_seq = Seq.Seq("ATGTCCGACCGCTCCGACCCGACGCACGCGATCTGGCAGAAGGTGCTCGCCGCCCTCACCGCGGACGACCGGATCACCCCGCAGCTGCACGGCTTCATCAGCCTCGTGGAGCCGAAGGGCGTGATGACCGGCACCCTCTACCTCGAGGTGCCCAACGACCTCACGCGGGGGATGCTCGAG") #REEMPLAZA LA SECUENCIA CON TU GEN dnaA!!
rev_seq = start_seq.reverse_complement()

# Archivos de entrada y salida
input_file = "/content/drive/MyDrive/Genomes/ensamblado/assembly.fasta"
output_file = "/content/drive/MyDrive/Genomes/ensamblado/genoma_rotado.fasta"

rotated_records = []
rotated_flag = False

for contig in SeqIO.parse(input_file, 'fasta'):
    if start_seq in contig.seq:
        idx = contig.seq.find(start_seq)
        new_seq = contig.seq[idx:] + contig.seq[:idx]
        rotated = SeqRecord(new_seq, id=contig.id, description="rotated")
        rotated_records.append(rotated)
        rotated_flag = True
    elif rev_seq in contig.seq:
        rc = contig.reverse_complement()
        idx = rc.seq.find(start_seq)
        new_seq = rc.seq[idx:] + rc.seq[:idx]
        rotated = SeqRecord(new_seq, id=contig.id, description="rotated (rev)")
        rotated_records.append(rotated)
        rotated_flag = True
    else:
        rotated_records.append(contig)

SeqIO.write(rotated_records, output_file, 'fasta')

# Confirmación visual
if rotated_flag:
    print("✅ Genoma rotado correctamente y guardado como 'genoma_rotado.fasta'")
else:
    print("⚠️ No se encontró la secuencia de dnaA. El genoma fue guardado sin rotar.")


## 🧬 Anotación del genoma con Bakta

**Bakta** es una herramienta de anotación genómica rápida, diseñada específicamente para bacterias. Detecta genes codificantes (CDS), tRNAs, rRNAs, elementos móviles, proteínas hipotéticas y más.

Produce salidas estándar como:
- `.gbff` → GenBank completo
- `.gff` → para pangenomas y visualización
- `.faa`, `.ffn`, `.tsv`, etc.

### 🧠 ¿Por qué es importante?

Una buena anotación te permite:
- Saber qué genes hay en tu genoma
- Compararlo con otros (p. ej. ANI, pangenomas)
- Encontrar genes de interés (resistencia, virulencia, etc.)

> ⚠️ **Debes tener acceso a la base de datos de Bakta**, que pesa varios GB. Se puede colocar en tu Google Drive personal y montar desde ahí.


In [ ]:
# 🔧 INSTALAR BAKTA (solo una vez)
!conda install -y -c bioconda -c conda-forge bakta=1.8.1


## 🧬 Descarga e instalación de la base de datos de Bakta

Bakta requiere una base de datos obligatoria para realizar la anotación de genomas. Esta base incluye proteínas, regiones conservadas y herramientas como AMRFinderPlus.

Hay dos versiones disponibles:
- `full`: para resultados completos (requiere >80 GB descomprimidos)
- `light`: más rápida y ligera (~4 GB), ideal para prácticas de clase

👉 En este curso usaremos la versión **light**.

Esta base de datos solo necesita descargarse una vez. Si la guardas en tu Google Drive, no tendrás que volver a bajarla.


In [ ]:
!bakta_db download --output /content/drive/MyDrive/bakta_db --type light
#SOLO SE CORRE UNA VEZ

In [ ]:
#Asegurate de reemplazar "/content/drive/MyDrive/Genomes/ensamblado/genoma_rotado.fasta" con el nombre/directorio de tu genoma rotado
# Asegurate de remplazar "/content/drive/MyDrive/Genomes/ensamblado/bakta_out" con el directorio de salida deseado para tus datos

!MPLBACKEND=Agg bakta \
  --db /content/drive/MyDrive/bakta_db/db-light \
  /content/drive/MyDrive/Genomes/ensamblado/genoma_rotado.fasta \
  --output /content/drive/MyDrive/Genomes/ensamblado/bakta_out \
  --prefix cepa01 \
  --threads 2

---

### 💡 ¿Problemas de espacio en Drive?

Si al intentar correr Bakta te aparece un error como:

```
ERROR: database file (pfam.h3m) not readable!
```

Es probable que **no tengas suficiente espacio en tu Google Drive** para alojar la base de datos completa. La versión *light* aún ocupa ~4 GB, y Google Colab necesita acceso completo para funcionar correctamente.

---

### 🌐 Solución alternativa: Usar el servidor web de Bakta

Si tu espacio en Drive es limitado, puedes **anotar tu genoma directamente en el servidor web oficial de Bakta**:

👉 [https://bakta.computational.bio](https://bakta.computational.bio)

Solo necesitas subir tu archivo `.fasta` rotado, y luego podrás **descargar los resultados y guardarlos manualmente** en tu carpeta de Google Drive.

📝 **Recomendación:** Renombra tus archivos de salida antes de subirlos al Drive para mantener un orden claro entre tus cepas (`cepa01.tsv`, `cepa01.gbk`, etc.).

---


## 🌐 Anotación de Genomas con Bakta (Web App)

Antes de realizar el análisis de pangenoma con Panaroo, necesitamos **anotar los genomas** para generar archivos `.gff` compatibles. En este curso usaremos la herramienta **Bakta** a través de su versión web.

🔗 Abre la herramienta en tu navegador:

👉 https://bakta.computational.bio/

### 📝 Instrucciones:

1. **Sube tu archivo `.fasta` rotado o ensamblado** (por ejemplo, `assembly.fasta`) en la sección correspondiente.
2. Marca la opción `--light` si te permite (versión más rápida).
3. Cambia el **nombre de la cepa** si lo deseas (opcional).
4. Haz clic en **Start annotation**.
5. Una vez terminado, descarga el archivo `.gff` que aparece en los resultados.
6. Guarda el archivo `.gff` en tu carpeta de Google Drive dentro de una carpeta llamada, por ejemplo:  
   `/content/drive/MyDrive/Genomes/gffs_panaroo/`

> 🔔 Asegúrate de usar **genomas rotados y ensamblados correctamente**, idealmente con un solo contig para mejorar la calidad de la anotación.

**Después de anotar todos los genomas, procederemos al análisis de pangenoma con Panaroo.**


## 🧬 Comparación Genómica con PyANI Plus (Colab)

**PyANI Plus** es una herramienta de Python para calcular la **Average Nucleotide Identity (ANI)** entre múltiples genomas. A diferencia de FastANI, PyANI Plus genera una **matriz de similitud completa** y también incluye visualizaciones, como mapas de calor (heatmaps), que facilitan la interpretación.

📌 Es ideal para comparar todos tus genomas al mismo tiempo y ver cómo se agrupan.


In [ ]:
pip install pyani-plus


### 📂 Paso 2: Preparar los genomas a comparar

Asegúrate de que todos tus archivos `.fasta` (idealmente ensamblados y anotados) estén en una misma carpeta, por ejemplo:

```
/content/drive/MyDrive/Genomes/EnsambladosFinales/
```

📁 Tu carpeta debería tener archivos como:

```
cepa01.fasta
cepa02.fasta
cepa03.fasta
...
```


In [ ]:
import matplotlib
matplotlib.use("Agg")


In [ ]:
# Apply MPLBACKEND=Agg to ensure matplotlib uses a non-interactive backend
!MPLBACKEND=Agg pyani-plus -h

In [ ]:
#Crea un archivo de texto con la ruta de tus genomas:

import os

# Directorio donde están los genomas ensamblados
genomes_dir = "/content/drive/MyDrive/Genomes/ensamblados_finales" #Cambia esta ruta por tu directorio

# Ruta del archivo de lista
fasta_list_path = "/content/genomes_list.txt"

# Crear archivo con rutas absolutas
with open(fasta_list_path, "w") as out:
    for fname in os.listdir(genomes_dir):
        if fname.endswith(".fasta"):
            full_path = os.path.join(genomes_dir, fname)
            out.write(full_path + "\n")

print(f"✅ Archivo de genomas generado en: {fasta_list_path}")


In [ ]:
import os

# Crear la carpeta si no existe
os.makedirs("/content/drive/MyDrive/Genomes/Resultados_ANI", exist_ok=True)


In [ ]:
# ✅ Ejecutar análisis ANI con sourmash en Google Colab
!MPLBACKEND=Agg pyani-plus sourmash /content/drive/MyDrive/Genomes/ensamblados_finales \
  --database /content/drive/MyDrive/Genomes/Resultados_ANI/ani_sourmash.db \
  --create-db \
  --name "ANI_sourmash_clase" \
  --executor local \
  --scaled 1000 \
  --kmersize 31


In [ ]:
# ⬇️ Generar el heatmap y matriz visual
!MPLBACKEND=Agg pyani-plus plot-run \
  --database /content/drive/MyDrive/Genomes/Resultados_ANI/ani_sourmash.db \
  --run-id 1 \
  --outdir /content/drive/MyDrive/Genomes/Resultados_ANI/plots_sourmash

# 📁 Se guardará una figura PNG y una matriz .csv visual en: plots_sourmash


## 🧬 Instalación de Panaroo en Google Colab

Para realizar el análisis de pangenoma, utilizaremos **Panaroo**, una herramienta eficiente para identificar genes ortólogos y generar matrices de presencia/ausencia.


In [ ]:
!conda install -c conda-forge -c bioconda panaroo


### 📁 Asegúrate de tener tus archivos `.gff` generados por Bakta o Prokka en una carpeta, por ejemplo:

```
/content/drive/MyDrive/Genomes/gff_panaroo/
├── cepa01.gff
├── cepa02.gff
├── ...

```

In [ ]:
!pip uninstall -y biopython
!pip install biopython==1.77

In [4]:
!mkdir -p /content/drive/MyDrive/Genomes/Panaroo_output #Crea un nuevo directorio para almacenar tus resultados

In [ ]:
import glob

# Buscar los archivos GFF
gff_files = glob.glob("/content/drive/MyDrive/Genomes/gff_panaroo/*.gff")

# Verifica que sí encontró archivos
print("Archivos GFF encontrados:")
print(gff_files)

# Une los archivos en un solo string
gff_input = " ".join(gff_files)


In [6]:
!sed -i "s/open(gff_file_name, 'rU')/open(gff_file_name, 'r')/" /usr/local/lib/python3.11/site-packages/panaroo/prokka.py
!find /usr/local/lib/python3.11/site-packages/panaroo/ -type f -name "*.py" -exec sed -i "s/'rU'/'r'/g" {} +
!find /usr/local/lib/python3.11/site-packages/panaroo/ -type f -name "*.py" -exec sed -i "s/'rU'/'r'/g" {} +
!sed -i "s/open(gff_handle_name, \"rU\")/open(gff_handle_name, \"r\")/" /usr/local/lib/python3.11/site-packages/panaroo/find_missing.py


In [ ]:
# Ejecuta panaroo con los archivos encontrados
!panaroo -i /content/drive/MyDrive/Genomes/gff_panaroo/*.gff \
         -o /content/drive/MyDrive/Genomes/Panaroo_output \
         --clean-mode strict \
         -t 4

#📊 Visualizando tu Pangenoma

## 🔍 ¿Qué es un UpSet Plot y cómo se aplica al estudio de pangenomas?

Un **UpSet Plot** es una herramienta de visualización que permite analizar **intersecciones entre conjuntos**. En lugar de usar diagramas de Venn, que se vuelven confusos cuando hay muchos grupos, el UpSet Plot representa las intersecciones de forma clara, escalable y cuantificable.

### 🧬 ¿Cómo se usa en genómica comparada?

En el contexto de un **pangenoma**, cada genoma analizado puede considerarse un "conjunto" de genes. El UpSet Plot nos ayuda a visualizar:

- Qué genes están **presentes en todos** los genomas (core genome).
- Qué genes están presentes **en algunos pero no todos** (shell genome).
- Qué genes son **únicos de un solo genoma** (cloud genes o singletons).

### 📁 ¿Qué archivo de Panaroo se utiliza?

El archivo que genera Panaroo y que vamos a usar se llama:
```
gene_presence_absence.Rtab

```

Este archivo contiene una **matriz binaria de presencia/ausencia**, donde:

- Cada **fila** representa un gen.
- Cada **columna** representa un genoma.
- El valor `1` indica que ese gen está presente en ese genoma.
- El valor `0` indica ausencia.

### 📊 ¿Qué veremos con el UpSet Plot?

- En el eje inferior: combinaciones de genomas que comparten ciertos genes.
- En el eje vertical: cuántos genes pertenecen a cada combinación.
- Barras adicionales indican cuántos genes tiene cada genoma individual.

Esta visualización nos permite interpretar rápidamente la **diversidad genética** entre los genomas comparados y entender el tamaño del **pangenoma**, el **genoma central** y el grado de **variabilidad entre cepas**.

---


In [ ]:
# Instalar librerías necesarias
!pip install -q upsetplot pandas matplotlib


In [18]:
# Importar librerías
import pandas as pd
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_indicators

In [ ]:
# Leer el archivo Rtab de Panaroo
path = "/content/drive/MyDrive/Genomes/Panaroo_output/gene_presence_absence.Rtab"
data = pd.read_csv(path, sep='\t')

# La primera columna es el nombre del gen
genes = data.iloc[:, 0]
presence_absence = data.iloc[:, 1:]

# Convertir valores distintos de 0 a True (presente), y 0 a False (ausente)
presence_absence = presence_absence != 0

# Convertir a formato compatible con UpSetPlot
upset_data = from_indicators(presence_absence)

# Graficar
plt.figure(figsize=(12, 6))
UpSet(upset_data, subset_size='count', show_counts=True).plot()
plt.show()


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Crear figura y plot
plt.figure(figsize=(12, 6))
plot = UpSet(upset_data, subset_size='count', show_counts=True)
plot.plot()

# Guardar como PNG en el mismo directorio (puedes ajustar ruta si usas Drive)
plt.savefig("/content/drive/MyDrive/Genomes/Panaroo_output/upset_plot.png", dpi=300, bbox_inches='tight')

# Mostrar el plot
plt.show()


## 📦 Parsnp: Genomic Core Alignment and Phylogenetic Tree Construction

**Parsnp** es una herramienta eficiente para alinear rápidamente genomas completos de bacterias con respecto a un genoma de referencia. Es útil para construir árboles filogenéticos basados en el alineamiento de los genes del **core genome**.

### 🧬 ¿Por qué Parsnp?

- Alinea genomas bacterianos completos de forma rápida y precisa.
- No requiere archivos de anotación (.gbk), aunque los puede utilizar si están disponibles.
- Produce directamente un **árbol filogenético** en formato Newick (`parsnp.tree`).
- Genera un archivo de alineamiento múltiple (`parsnp.xmfa`) y un archivo visualizable con Gingr (`parsnp.ggr`).

### 🚀 En este flujo de trabajo:

- Usamos un conjunto de genomas ensamblados (`.fasta`) localizados en una carpeta.
- Elegimos uno como genoma de **referencia**.
- Ejecutamos Parsnp para alinear todos los genomas respecto al de referencia.
- Obtenemos un árbol filogenético y archivos de alineamiento del core genome.

### 📁 Estructura esperada del directorio:



```
ensamblados_finales/
├── NCPPB2581.fasta         <- Genoma de referencia
├── CN11.fasta              <- Otros genomas
├── CN12.fasta
├── ...
```

---


In [ ]:
#🔧 Instalación de Parsnp en Google Colab

!apt-get install -y libncurses5
!wget https://github.com/marbl/parsnp/releases/download/v1.2/parsnp-Linux64-v1.2.tar.gz
!tar -xvzf parsnp-Linux64-v1.2.tar.gz
!mv Parsnp-Linux64-v1.2/parsnp /usr/local/bin/


In [32]:
# Verifica la instalación
!parsnp --version

In [ ]:
# Crear carpeta de salida (si no existe)
!mkdir -p /content/drive/MyDrive/Genomes/parsnp_output

# Ejecutar Parsnp con tu genoma de referencia y todos los demás
!parsnp -r /content/drive/MyDrive/Genomes/ensamblados_finales/NCPPB2581.fasta \
        -d /content/drive/MyDrive/Genomes/ensamblados_finales/ \
        -o /content/drive/MyDrive/Genomes/parsnp_output \
        -p 4


📌 **Notas**:
- `-r` es el genoma de referencia (debe ser uno de los archivos `.fasta`).
- `-d` es la ruta a la **carpeta** donde están todos los `.fasta`.
- `-o` es la carpeta donde se guardarán los resultados.
- `-p` indica el número de hilos (`threads`).

## 🌳 Visualización del Árbol Filogenético en iTOL (Interactive Tree of Life)

[iTOL](https://itol.embl.de/) es una plataforma web interactiva para visualizar, anotar y compartir árboles filogenéticos de manera intuitiva y estética. Puedes subir árboles en formato **Newick (.tree)** generado por Parsnp u otras herramientas.

### 📥 ¿Cómo subir tu árbol a iTOL?

1. Ve a [https://itol.embl.de/](https://itol.embl.de/)
2. Inicia sesión o crea una cuenta gratuita si no tienes una.
3. Haz clic en **"Upload Tree"** (Subir árbol).
4. Selecciona el archivo `parsnp.tree` generado por Parsnp.
5. Da clic en **"Upload Tree"** para cargarlo.

### 🖌️ Opciones de personalización:

- Cambia los colores de ramas o etiquetas por grupo.
- Añade anotaciones (anillos, calor, formas).
- Exporta la figura como imagen (.png, .svg) o PDF.

### 🧪 Tips útiles:

- Asegúrate de que los nombres de los genomas en el árbol coincidan con tus datos experimentales.
- Puedes crear un archivo adicional con metadatos para colorear ramas automáticamente.

## 🎨 Anotaciones adicionales en iTOL (opcional)

Además del árbol filogenético, iTOL te permite subir archivos de anotaciones para agregar colores, etiquetas, formas u otras visualizaciones que enriquezcan la interpretación del árbol.

Estos archivos pueden:
- Colorear ramas por país o estado
- Agregar etiquetas con el año de aislamiento, grupo, patotipo, etc.
- Visualizar valores como intensidad de pigmento, presencia/ausencia de genes, etc.

### 📁 Ejemplo de archivo de anotación para colorear ramas por país

```text
DATASET_COLORSTRIP
SEPARATOR TAB
DATASET_LABEL	País de origen
COLOR	#ff0000

LEGEND_TITLE	País
LEGEND_SHAPES	1	1
LEGEND_COLORS	#ff0000	#0000ff
LEGEND_LABELS	México	EE.UU.

DATA
CN11	#ff0000
CN12	#ff0000
NCPPB2581	#0000ff


#✅ ¿Terminaste tus análisis? ¡Aquí hay más herramientas para explorar tu genoma!


### 🧪 AntiSMASH – Detección de Clústeres Biosintéticos (BGCs)

[https://antismash.secondarymetabolites.org](https://antismash.secondarymetabolites.org)

AntiSMASH permite predecir **clústeres biosintéticos de metabolitos secundarios**, como bacteriocinas, sideróforos, pigmentos y antibióticos. Es ideal para descubrir funciones ecológicas de tus bacterias.

**¿Qué necesitas?**
- Subir un archivo `.fasta` de tu genoma ensamblado (de preferencia en contigs largos o ensamblado cerrado)
- (Opcional) Subir archivo `.gbk` para usar anotaciones

📍 *Pro tip:* Puedes subir varios genomas en lote si los juntas en un ZIP.

---

### 🧬 dbCAN – Análisis de CAZymes (enzimas que degradan carbohidratos)

[http://bcb.unl.edu/dbCAN2/](http://bcb.unl.edu/dbCAN2/)

dbCAN2 predice **CAZymes** (Carbohydrate-Active enZymes), como celulasas, quitinasas y amilasas, que son esenciales para degradar o modificar polisacáridos, como la pared celular vegetal.

**¿Qué necesitas?**
- Subir un archivo `.faa` (proteínas en formato FASTA) generado por Bakta o Prokka

🔎 También te da acceso a una tabla interactiva con clasificación por familia CAZy (GH, GT, CE, etc.)

---

### 🔧 ¿Otras herramientas recomendadas?

| Herramienta        | ¿Para qué sirve?                                                | Enlace                                      |
|--------------------|------------------------------------------------------------------|---------------------------------------------|
| **SignalP (online)** | Predicción de péptidos señal para secreción                    | https://services.healthtech.dtu.dk/service.php?SignalP-6.0 |
| **VFDB / VFanalyzer** | Identificación de factores de virulencia                     | http://www.mgc.ac.cn/cgi-bin/VFs/v5/main.cgi |
| **IslandViewer**   | Detección de islas genómicas (posible HGT)                      | https://www.pathogenomics.sfu.ca/islandviewer/ |

---

✨ ¡Estas herramientas te ayudan a descubrir el potencial ecológico, funcional y evolutivo de tus bacterias! ¿Quieres integrar alguna de ellas al flujo de trabajo en Colab?
